# DPO Polish: Preference-Based Reasoning Quality Improvement

This notebook applies **Iterative DPO** (Direct Preference Optimization) to polish
reasoning quality after KTO Socratic alignment.

**Training pipeline stage:** 3 of 3 (GSPO (triple reward) → KTO → **DPO**)

**Target hardware:** Google Colab A100 80GB / H100 80GB (auto-detect)

**Key features:**
- Loads KTO-aligned model (Socratic style already learned)
- Generate preference pairs: chosen (correct + best format) vs rejected (worst format/incorrect)
- DPO training with beta=0.1 for conservative policy update
- Guard: skip DPO if < MIN_PAIRS preference pairs (model already strong)

**Evaluation:** Removed from notebook — run locally via `training/scripts/evaluate_stage.py`

**References:**
- [DPO (arXiv 2305.18290)](https://arxiv.org/abs/2305.18290) — Direct Preference Optimization
- [KTO (arXiv 2402.01306)](https://arxiv.org/abs/2402.01306) — Kahneman-Tversky Optimization (previous stage)
- [Iterative DPO (arXiv 2503.12854)](https://arxiv.org/abs/2503.12854) — Self-play preference learning

In [ ]:
# Disable gradient offloading BEFORE importing unsloth
import os
os.environ["UNSLOTH_OFFLOAD_GRADIENTS"] = "0"

# ============================================================
# ALL dependencies in one shot. After this cell: RESTART RUNTIME.
# After restart: SKIP this cell, start from Cell 2.
# ============================================================

# Step 1: Unsloth
!pip install -q --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo

# Step 2: Transformers v5 (Qwen3.5 hybrid arch) + ML
!pip install -q "transformers>=5.0.0" trl peft datasets

# Step 3: Fix Colab packages broken by numpy upgrade
!pip install -q --upgrade scipy torchvision "Pillow<12.0"

# Step 4: Other deps
!pip install -q accelerate bitsandbytes sentencepiece protobuf
!pip install -q sympy chempy

from huggingface_hub import login
login()

print()
print("=" * 60)
print("  RESTART RUNTIME NOW: Runtime -> Restart session")
print("  After restart: SKIP this cell, run Cell 2 onwards.")
print("=" * 60)

In [ ]:
# ============================================================
# Add Drive root to sys.path (training/ is at MyDrive level)
# ============================================================
import sys, os

DRIVE_ROOT = "/content/drive/MyDrive"
if DRIVE_ROOT not in sys.path:
    sys.path.insert(0, DRIVE_ROOT)

for s in ["training/scripts/stem_rewards.py", "training/scripts/verify_answers.py", "training/scripts/sort_curriculum.py"]:
    print(f"  {'OK' if os.path.exists(os.path.join(DRIVE_ROOT, s)) else 'MISSING'} {s}")

In [ ]:
# ============================================================
# Configuration
# ============================================================
import torch

# Model — Instruct base gives dialogue abilities built-in
BASE_MODEL = "Qwen/Qwen3.5-9B"
KTO_CHECKPOINT = "/content/drive/MyDrive/MITS/checkpoints/kto_qwen3.5_9b/final"
KTO_HF_REPO = "Siesher/mits-qwen3-9b-kto"
OUTPUT_DIR = "/content/drive/MyDrive/MITS/checkpoints/dpo_qwen3.5_9b"

# ---- Auto-detect GPU: A100 / H100 / G4 (RTX PRO 6000 Blackwell) ----
_gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
_gpu_mem_gb = torch.cuda.get_device_properties(0).total_mem / (1024**3) if torch.cuda.is_available() else 0

if "RTX PRO 6000" in _gpu_name or ("Blackwell" in _gpu_name and _gpu_mem_gb >= 90):
    GPU_TYPE = "G4 (RTX PRO 6000 Blackwell 96GB)"
    PAIRS_PER_PROBLEM = 16     # 96GB: max completions for best pair diversity
    DPO_BATCH_SIZE = 4
    DPO_GRAD_ACCUM = 4        # effective batch = 16
elif "H100" in _gpu_name:
    GPU_TYPE = "H100"
    PAIRS_PER_PROBLEM = 12
    DPO_BATCH_SIZE = 4
    DPO_GRAD_ACCUM = 4        # effective batch = 16
elif "A100" in _gpu_name and _gpu_mem_gb >= 70:
    GPU_TYPE = "A100-80GB"
    PAIRS_PER_PROBLEM = 8
    DPO_BATCH_SIZE = 2
    DPO_GRAD_ACCUM = 8        # effective batch = 16
elif _gpu_mem_gb >= 40:
    GPU_TYPE = f"Generic ({_gpu_name})"
    PAIRS_PER_PROBLEM = 4
    DPO_BATCH_SIZE = 1
    DPO_GRAD_ACCUM = 16
else:
    raise RuntimeError(
        f"GPU {_gpu_name} has only {_gpu_mem_gb:.0f}GB VRAM. "
        f"Disconnect and try again for a better GPU."
    )

print(f"Detected GPU: {_gpu_name} ({_gpu_mem_gb:.0f} GB) -> preset: {GPU_TYPE}")

# DPO parameters
DPO_BETA = 0.1                 # Conservative policy update
DPO_LR = 5e-7                  # Very low LR for polish
DPO_STEPS = 200
DPO_WARMUP_RATIO = 0.1
MIN_PAIRS = 50                 # Skip DPO if too few valid pairs

# Generation
MAX_COMPLETION = 1024
MAX_PROMPT_LENGTH = 512
MAX_SEQ_LENGTH = MAX_PROMPT_LENGTH + MAX_COMPLETION

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.0
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

DOMAINS = ["math", "physics", "chemistry", "biology", "cs"]
SYSTEM_PROMPT = "Ты — репетитор по STEM. Реши задачу пошагово и запиши финальный ответ в \\boxed{}."

print(f"Hardware: {GPU_TYPE}")
print(f"Base model: {BASE_MODEL}")
print(f"DPO: beta={DPO_BETA}, lr={DPO_LR}, steps={DPO_STEPS}")
print(f"Pairs per problem: {PAIRS_PER_PROBLEM}, min pairs: {MIN_PAIRS}")

In [ ]:
# ============================================================
# Mount Drive, resolve checkpoint, load data
# ============================================================
import json
import os
import re
import random
from collections import Counter, defaultdict

DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = True
    print("Google Drive mounted successfully")
except Exception as e:
    print(f"Drive mount failed ({e}), using local storage")
    OUTPUT_DIR = "/content/checkpoints/dpo_qwen3.5_9b"

# Resolve KTO checkpoint: Drive first, then download from HuggingFace
if os.path.exists(KTO_CHECKPOINT):
    print(f"KTO checkpoint found: {KTO_CHECKPOINT}")
elif KTO_HF_REPO:
    from huggingface_hub import snapshot_download
    KTO_CHECKPOINT = snapshot_download(KTO_HF_REPO)
    print(f"Downloaded KTO adapter from HuggingFace to: {KTO_CHECKPOINT}")
else:
    raise FileNotFoundError(f"KTO checkpoint not found: {KTO_CHECKPOINT}")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- Load RL problems: Drive JSONL → HF "rl" → HF "gspo" fallback ----
from datasets import load_dataset

RL_DATA_PATH = "/content/drive/MyDrive/training/data/rl_combined.jsonl"

if os.path.exists(RL_DATA_PATH):
    print(f"Loading RL data from Drive: {RL_DATA_PATH}")
    problems = []
    with open(RL_DATA_PATH, "r", encoding="utf-8") as f:
        for line in f:
            problems.append(json.loads(line))
    print(f"Loaded {len(problems)} from Drive JSONL")
else:
    try:
        print("Trying HF 'rl' config...")
        hf_ds = load_dataset("Siesher/mits-stem-training-data", "rl")
        problems = [dict(r) for r in hf_ds["train"]]
        if "test" in hf_ds:
            problems += [dict(r) for r in hf_ds["test"]]
        print(f"Loaded {len(problems)} from HF 'rl' config")
    except Exception:
        print("Falling back to HF 'gspo' config...")
        hf_ds = load_dataset("Siesher/mits-stem-training-data", "gspo")
        problems = [dict(r) for r in hf_ds["train"]] + [dict(r) for r in hf_ds["test"]]
        print(f"Loaded {len(problems)} from HF 'gspo' (fallback)")

verifiable_problems = [
    p for p in problems
    if p.get("type", "verifiable") == "verifiable"
    and p.get("answer_type", "numeric") != "conceptual"
]
print(f"Loaded {len(verifiable_problems)} verifiable problems")

In [ ]:
# ============================================================
# Load model + KTO adapter (auto-detect PEFT)
# ============================================================
import torch
from unsloth import FastLanguageModel

# Use from_pretrained with adapter repo — Unsloth reads adapter_config.json,
# loads base model, applies adapter + optimizations automatically.
_kto_source = KTO_HF_REPO if KTO_HF_REPO else KTO_CHECKPOINT
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=_kto_source,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    dtype=torch.bfloat16,
)

if hasattr(model, 'peft_config'):
    _cfg = list(model.peft_config.values())[0]
    effective_r = _cfg.r
    kto_alpha = _cfg.lora_alpha
    print(f"KTO adapter loaded from {_kto_source}: r={effective_r}, alpha={kto_alpha}")
else:
    effective_r = LORA_R
    kto_alpha = LORA_ALPHA
    print(f"WARNING: peft_config not found after loading {_kto_source}")


# Qwen3.5 returns Processor (multimodal), unwrap to tokenizer
if not hasattr(tokenizer, "vocab_size") and hasattr(tokenizer, "tokenizer"):
    _processor = tokenizer
    tokenizer = _processor.tokenizer
    print(f"Unwrapped Qwen3VLProcessor -> {type(tokenizer).__name__}")

model.gradient_checkpointing_enable()

if hasattr(model, 'hf_device_map'):
    model.hf_device_map = {'': 0}

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,}")

# Qwen3.5 text-only fix: replace compute_3d_position_ids entirely
# NOTE: **kwargs required — transformers v5 passes mm_token_type_ids, cache_position, etc.
import torch as _torch
_qwen_inner = model
while hasattr(_qwen_inner, 'model'):
    _qwen_inner = _qwen_inner.model

def _text_only_pos_ids(self, input_ids=None, inputs_embeds=None,
                        image_grid_thw=None, video_grid_thw=None,
                        attention_mask=None, past_key_values=None,
                        **kwargs):
    if inputs_embeds is not None:
        bs, sl = inputs_embeds.shape[:2]
        dev = inputs_embeds.device
    else:
        bs, sl = input_ids.shape
        dev = input_ids.device
    past = 0
    if past_key_values is not None and hasattr(past_key_values, 'get_seq_length'):
        try: past = past_key_values.get_seq_length()
        except: past = 0
    if attention_mask is not None and past == 0:
        pos = attention_mask.long().cumsum(-1) - 1
        pos.masked_fill_(attention_mask == 0, 1)
        pos = pos.unsqueeze(0).expand(3, -1, -1)
    else:
        pos = _torch.arange(past, past + sl, device=dev).view(1, 1, -1).expand(3, bs, -1)
    self.rope_deltas = _torch.zeros(bs, 1, dtype=_torch.long, device=dev)
    return pos

type(_qwen_inner).compute_3d_position_ids = _text_only_pos_ids
print(f"compute_3d_position_ids replaced (text-only, **kwargs-safe) on {type(_qwen_inner).__name__}")

In [ ]:
# ============================================================
# Verification + format scoring
# ============================================================

_verify_imported = False
try:
    from training.scripts.verify_answers import verify, extract_answer
    _verify_imported = True
    print("Imported verify functions")
except ImportError:
    try:
        import sys
        sys.path.insert(0, "/content/drive/MyDrive")
        from training.scripts.verify_answers import verify, extract_answer
        _verify_imported = True
    except ImportError:
        import sympy
        def extract_answer(text):
            if "</think>" in text: text = text.split("</think>")[-1].strip()
            boxed = re.findall(r'\\boxed\{([^}]+)\}', text)
            return boxed[-1].strip() if boxed else text.strip()


def verify_completion(completion, problem):
    answer = extract_answer(completion)
    truth = problem.get("ground_truth", problem.get("answer", ""))
    domain = problem.get("domain", "math")
    if _verify_imported:
        result = verify(answer=answer, truth=truth, domain=domain,
                       question_type=problem.get("type", "calc"),
                       test_cases=problem.get("test_cases"))
        return result.correct
    return answer.strip().lower() == str(truth).strip().lower()


def score_format(text):
    """Score reasoning format quality [0, 1]."""
    score = 0.0
    if "\\boxed{" in text: score += 0.4
    step_markers = ["step", "therefore", "thus", "hence", "because",
                    "\u0448\u0430\u0433", "\u0441\u043b\u0435\u0434\u043e\u0432\u0430\u0442\u0435\u043b\u044c\u043d\u043e",
                    "\u0437\u043d\u0430\u0447\u0438\u0442", "\u043f\u043e\u0442\u043e\u043c\u0443 \u0447\u0442\u043e",
                    "\u0434\u0430\u043b\u0435\u0435", "\u043f\u043e\u0434\u0441\u0442\u0430\u0432\u0438\u043c"]
    if any(m in text.lower() for m in step_markers): score += 0.3
    word_count = len(text.split())
    if 50 < word_count < 800: score += 0.2
    if "<think>" in text and "</think>" in text: score += 0.1
    return min(1.0, score)


print("Verification and format scoring ready")

In [ ]:
# ============================================================
# Build preference pairs: chosen (correct + best format) vs rejected (incorrect/worst)
# Self-play: generate multiple completions, pick best correct and worst incorrect
# ============================================================

import random
from collections import defaultdict

print("Generating preference pairs (self-play)...")
FastLanguageModel.for_inference(model)

preference_pairs = []
pair_stats = {
    "total_problems_attempted": 0,
    "chosen_generated": 0,
    "rejected_generated": 0,
    "problems_with_pairs": 0,
}

# Sample problems for DPO
dpo_sample_size = min(len(verifiable_problems), 500)
dpo_problems = random.sample(verifiable_problems, dpo_sample_size)

for i, problem in enumerate(dpo_problems):
    pair_stats["total_problems_attempted"] += 1
    domain = problem.get("domain", "math")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem["prompt"]},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=True,
    )
    inputs = tokenizer(
        prompt_text, return_tensors="pt",
        truncation=True, max_length=MAX_PROMPT_LENGTH
    ).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    # Generate PAIRS_PER_PROBLEM completions
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=MAX_COMPLETION,
            temperature=0.9, do_sample=True,
            num_return_sequences=PAIRS_PER_PROBLEM,
        )

    # Classify as correct/incorrect + score format
    correct_comps = []
    incorrect_comps = []
    for j in range(outputs.shape[0]):
        comp = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True)
        is_correct = verify_completion(comp, problem)
        fmt_score = score_format(comp)

        if is_correct:
            correct_comps.append((comp, fmt_score))
        else:
            incorrect_comps.append((comp, fmt_score))

    if not correct_comps:
        continue

    # Pick best correct as "chosen" (highest format score)
    correct_comps.sort(key=lambda x: x[1], reverse=True)
    chosen_comp = correct_comps[0][0]
    pair_stats["chosen_generated"] += 1

    # Pick worst incorrect as "rejected"
    rejected_comp = None
    if incorrect_comps:
        incorrect_comps.sort(key=lambda x: x[1])
        rejected_comp = incorrect_comps[0][0]
        pair_stats["rejected_generated"] += 1

    if rejected_comp is None:
        continue

    pair_stats["problems_with_pairs"] += 1

    # Format as DPO preference pair
    chosen_messages = messages + [{"role": "assistant", "content": chosen_comp}]
    rejected_messages = messages + [{"role": "assistant", "content": rejected_comp}]

    preference_pairs.append({
        "prompt": prompt_text,
        "chosen": tokenizer.apply_chat_template(
            chosen_messages, tokenize=False, add_generation_prompt=False,
            enable_thinking=True,
        ),
        "rejected": tokenizer.apply_chat_template(
            rejected_messages, tokenize=False, add_generation_prompt=False,
            enable_thinking=True,
        ),
    })

    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{dpo_sample_size}: {len(preference_pairs)} pairs")

FastLanguageModel.for_training(model)

dpo_loss = None

print(f"\nPreference pair generation complete:")
print(f"  Total pairs: {len(preference_pairs)}")
print(f"  Chosen generated: {pair_stats['chosen_generated']}")
print(f"  Rejected generated: {pair_stats['rejected_generated']}")
print(f"  Min pairs threshold: {MIN_PAIRS}")


In [ ]:
# ============================================================
# DPO Training (with guard + Colab disconnect recovery)
# ============================================================
# Fix: llm_blender uses removed TRANSFORMERS_CACHE (transformers v5)
import transformers.utils.hub as _tf_hub
if not hasattr(_tf_hub, 'TRANSFORMERS_CACHE'):
    try:
        from huggingface_hub.constants import HF_HUB_CACHE
        _tf_hub.TRANSFORMERS_CACHE = HF_HUB_CACHE
    except ImportError:
        _tf_hub.TRANSFORMERS_CACHE = '/root/.cache/huggingface/hub'


from trl import DPOConfig, DPOTrainer
from datasets import Dataset


def find_latest_checkpoint(output_dir):
    """Find latest TRL checkpoint for resume after Colab disconnect."""
    if not os.path.isdir(output_dir):
        return None
    checkpoints = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
    if not checkpoints:
        return None
    latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
    path = os.path.join(output_dir, latest)
    print(f"  Found checkpoint: {path}")
    return path


dpo_skipped = False

if len(preference_pairs) < MIN_PAIRS:
    print(f"\nWARNING: Only {len(preference_pairs)} pairs < MIN_PAIRS={MIN_PAIRS}")
    print("Skipping DPO — KTO checkpoint is already strong enough.")
    print("The KTO adapter will be used as the final model.")
    dpo_skipped = True
else:
    print(f"\nProceeding with DPO training on {len(preference_pairs)} preference pairs")

    # Format as HuggingFace Dataset
    dpo_dataset = Dataset.from_list(preference_pairs)
    print(f"DPO dataset: {len(dpo_dataset)} examples")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    dpo_config = DPOConfig(
        output_dir=OUTPUT_DIR,
        max_steps=DPO_STEPS,
        per_device_train_batch_size=DPO_BATCH_SIZE,
        gradient_accumulation_steps=DPO_GRAD_ACCUM,
        learning_rate=DPO_LR,
        lr_scheduler_type="cosine",
        warmup_ratio=DPO_WARMUP_RATIO,
        beta=DPO_BETA,
        max_length=MAX_SEQ_LENGTH,
        max_prompt_length=MAX_PROMPT_LENGTH,
        bf16=True,
        logging_steps=10,
        save_steps=100,
        save_total_limit=2,
        optim="adamw_torch_fused",
        seed=42,
        report_to="none",
    )

    trainer = DPOTrainer(
        model=model,
        args=dpo_config,
        train_dataset=dpo_dataset,
        processing_class=tokenizer,
    )

    # Resume from checkpoint if Colab session was interrupted
    dpo_resume = find_latest_checkpoint(OUTPUT_DIR)
    print("Starting DPO training...")
    result = trainer.train(resume_from_checkpoint=dpo_resume)
    dpo_loss = result.training_loss
    print(f"DPO training complete! Loss: {dpo_loss:.4f}")

    trainer.save_model(os.path.join(OUTPUT_DIR, "final"))
    del trainer
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# Save final adapter + metrics + push to HuggingFace
# ============================================================

final_adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)
print(f"Final DPO adapter saved to {final_adapter_path}")

# Save metrics
eval_metrics = {
    "stage": "dpo" if not dpo_skipped else "dpo_skipped",
    "pipeline_position": "3 of 3",
    "dpo_skipped": dpo_skipped,
    "final_accuracy": final_acc,
    "final_format_score": final_fmt,
    "domain_results": final_domain_metrics,
    "preference_pairs_generated": len(preference_pairs),
    "pair_generation_stats": pair_stats,
    "dpo_loss": dpo_loss if not dpo_skipped else None,
}
eval_path = os.path.join(OUTPUT_DIR, "dpo_eval_metrics.json")
with open(eval_path, "w") as f:
    json.dump(eval_metrics, f, indent=2, default=str)
print(f"Eval metrics saved to {eval_path}")

# Save config
config = {
    "stage": "dpo",
    "pipeline": "GSPO (triple reward) -> KTO -> DPO",
    "pipeline_position": "3 of 3",
    "base_model": BASE_MODEL,
    "kto_source": _kto_source,
    "hardware": f"A100 {A100_VRAM_GB}GB",
    "dpo_beta": DPO_BETA,
    "dpo_lr": DPO_LR,
    "dpo_steps": DPO_STEPS,
    "pairs_per_problem": PAIRS_PER_PROBLEM,
    "min_pairs": MIN_PAIRS,
    "dpo_skipped": dpo_skipped,
    "lora_r": effective_r,
    "lora_alpha": kto_alpha,
    "lora_dropout": LORA_DROPOUT,
    "total_problems": len(verifiable_problems),
    "preference_pairs": len(preference_pairs),
    "references": [
        "DPO arXiv:2305.18290",
        "Iterative DPO arXiv:2503.12854",
    ],
}
config_path = os.path.join(OUTPUT_DIR, "training_config.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print(f"Config saved to {config_path}")

# Push to HuggingFace
PUSH_TO_HUB = True
HF_REPO_ID = "Siesher/mits-qwen3-9b-final"

if PUSH_TO_HUB:
    model.push_to_hub(HF_REPO_ID, private=True)
    tokenizer.push_to_hub(HF_REPO_ID, private=True)
    print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")

print("\n" + "="*60)
print("TRAINING PIPELINE COMPLETE")
print("="*60)
print(f"Pipeline: GSPO (triple reward) -> KTO -> DPO")
print(f"Final model: {final_adapter_path}")
print(f"Overall accuracy: {100*final_acc:.1f}%")
print(f"Format quality: {final_fmt:.3f}")
print(f"\nHuggingFace repos:")
print(f"  Stage 1 (GSPO):   Siesher/mits-qwen3-9b-gspo")
print(f"  Stage 2 (KTO):    Siesher/mits-qwen3-9b-kto")
print(f"  Stage 3 (DPO):     Siesher/mits-qwen3-9b-final")
print("\nThe final adapter can be exported to GGUF for Ollama deployment.")

In [ ]:
# ============================================================
# Evaluation removed — run locally:
#   python training/scripts/evaluate_stage.py --stage dpo \
#     --adapter /content/drive/MyDrive/MITS/checkpoints/dpo_qwen3.5_9b/final
# ============================================================
print("Post-training evaluation skipped (run locally)")